# Урок 12 - Сокращение истории чата с помощью Agent Scratchpad

В этой тетрадке демонстрируется, как управлять контекстом в длинных разговорах с использованием Microsoft Agent Framework. По мере роста разговоров увеличивается количество токенов — в конечном итоге превышая окно контекста модели. Мы решаем эту проблему с помощью **шаблона суммирования контекста** и **agent scratchpad** для постоянной памяти.

## Чему вы научитесь:
1. **Почему важно управление контекстом**: понимание ограничений токенов и окон контекста
2. **Агенты, учитывающие контекст**: создание агентов, которые управляют своим собственным контекстом разговора
3. **Шаблон суммирования контекста**: использование инструментов для сокращения истории разговоров
4. **Agent Scratchpad**: постоянная память, сохраняющаяся при сокращении контекста

## Предварительные требования:
- Настройка Azure OpenAI с конфигурированными переменными среды
- Понимание базовых концепций агентов из предыдущих уроков


## Настройка


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Почему управление контекстом важно

Каждый LLM имеет конечное **окно контекста** — максимальное количество токенов, которое он может обработать за один запрос. По мере развития многократного диалога:

- **Количество токенов растёт линейно** с каждым сообщением пользователя и ответом ассистента.
- **Токены подсказки определяют стоимость**, потому что вся история пересылается заново каждый ход.
- В итоге разговор **превышает окно контекста**, и модель либо усекает, либо выдаёт ошибку.

### Стратегии управления контекстом

| Стратегия | Как работает | Компромисс |
|---|---|---|
| **Усечение** | Удаление самых старых сообщений | Потеря раннего контекста |
| **Резюмирование** | Сжатие старых сообщений в краткое содержание | Некоторая потеря деталей, но ключевые моменты сохраняются |
| **Черновик / Внешняя память** | Хранение ключевых фактов вне разговора | Требует вызова инструментов, но сохраняется при любом сокращении |

В этой тетрадке мы комбинируем **резюмирование** с **инструментом черновика**, чтобы агент сохранял непрерывность даже при сжатии истории разговора.


## Создание агента, учитывающего контекст


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Моделирование длительного разговора

Давайте пройдем через многоэтапный разговор, чтобы увидеть, как накапливается контекст. Агент должен сохранять ключевые детали (предпочтения, бюджет, даты путешествия) на протяжении всех этапов и демонстрировать непрерывность.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Обратите внимание, как агент сохраняет контекст из предыдущих реплик — он знает о Японии, суши, храмах, фотографии, бюджете в 3000 долларов, самостоятельных путешествиях и временных рамках апреля. В коротком разговоре это работает хорошо, но по мере роста беседы полная история становится слишком дорогой для повторной отправки.

Давайте продолжим разговор с большим количеством реплик, чтобы увидеть накопление контекста:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Шаблон суммирования контекста

По мере развития разговора мы можем использовать **инструмент суммирования**, чтобы сжать накопленный контекст в компактный формат. Агент вызывает этот инструмент, чтобы зафиксировать ключевые предпочтения, так что даже если старые сообщения будут удалены, важная информация сохранится.

Этот шаблон является строительным блоком для более сложного сокращения истории:
1. Агент выявляет ключевые факты из разговора
2. Он вызывает инструмент суммирования для их сохранения
3. Старые сообщения можно безопасно удалить, потому что в резюме зафиксировано то, что важно

Ниже мы определяем инструмент `summarize_preferences`, который агент может вызвать, чтобы записать компактное резюме того, что он узнал.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Резюме

В этом уроке вы узнали, как управлять контекстом в длительных разговорах с агентом с помощью Microsoft Agent Framework:

### Основные концепции
- **Окна контекста ограничены** — каждый токен в истории разговора стоит денег и учитывается в лимите.
- **Инструменты суммирования** позволяют агенту сокращать накопленный контекст в компактные резюме, уменьшая использование токенов при сохранении важной информации.
- **Временные записные книжки агента** предоставляют постоянную внешнюю память, которая сохраняется при любом сокращении разговора.

### Что вы создали
- **Агент с учетом контекста**, который поддерживает непрерывность в многоходовых разговорах
- **Инструмент суммирования** (`summarize_preferences`), который фиксирует ключевые детали пользователя в компактном формате
- **Многоходовой разговор**, демонстрирующий сохранение контекста и обработку изменений

### Применение в реальном мире
- **Боты службы поддержки**: запоминают предпочтения на протяжении долгих сессий поддержки
- **Персональные помощники**: отслеживают текущие проекты без необходимости повторного объяснения контекста
- **Образовательные наставники**: сохраняют прогресс студентов на разных этапах взаимодействия

### Следующие шаги
- Реализовать полноценный инструмент записной книжки с файловым сохранением данных
- Добавить автоматическое усечение истории после суммирования
- Сочетать с векторными базами данных для семантического поиска по памяти
- Создавать агентов, которые могут продолжать разговоры через несколько дней с полным контекстом


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
